# XSD to Synthetic Data Framework

Generate synthetic IRS tax form data from XSD schemas.

**Default Configuration:**
- Form: IRS 1120 (U.S. Corporation Income Tax Return)
- Catalog: `edp.sandbox_bronze.irs_synthetic`
- Rows: 1,000

**Quick Start:** Click "Run All"


## Step 1: Import Library

In [ ]:
%run "./XSD to Synthetic - Library"

## Step 2: Install Packages

In [ ]:
# Install required packages
%pip install -q xmlschema dbldatagen faker


## Step 3: Configuration

In [ ]:
# Configuration widgets
try:
    # === ESSENTIAL CONFIGURATION ===

    # 1. Form Type
    dbutils.widgets.dropdown('form_type', '1120',
        ['990', '990-EZ', '990-T', '1040', '1120', 'Custom'],
        'Form Type')

    # 2. Infrastructure
    dbutils.widgets.text('volume_base_path',
        '/Volumes/edp/sandbox_bronze/sandbox_volume/xsds/',
        'XSD Volume Base Path')

    # 3. Output Location
    dbutils.widgets.text('catalog', 'edp', 'Output Catalog')
    dbutils.widgets.text('schema', 'sandbox_bronze', 'Output Schema')
    dbutils.widgets.text('table', 'irs_synthetic', 'Output Table')

    # 4. Data Generation
    dbutils.widgets.text('num_rows', '1000', 'Number of Rows')

    # === ADVANCED (only for Custom form type) ===
    dbutils.widgets.text('xsd_path_override', '',
        'Custom XSD Path (leave blank for presets)')

    print("Widgets configured")

except NameError:
    # Running locally without dbutils
    print("Running without dbutils - using default configuration")
    class MockDbutils:
        class Widgets:
            @staticmethod
            def get(key):
                defaults = {
                    'form_type': '1120',
                    'volume_base_path': '/Volumes/edp/sandbox_bronze/sandbox_volume/xsds/',
                    'catalog': 'edp',
                    'schema': 'sandbox_bronze',
                    'table': 'irs_synthetic',
                    'num_rows': '1000',
                    'xsd_path_override': ''
                }
                return defaults.get(key, '')
        widgets = Widgets()
    dbutils = MockDbutils()


In [ ]:
# ============================================================
# XSD CONFIGURATION
# ============================================================

# Get volume base from widget (infrastructure-specific)
XSD_VOLUME_PATH = dbutils.widgets.get('volume_base_path')

# XSD version (change when IRS releases new schema version)
XSD_VERSION = '2024v5.5'

# Combine into full base path
XSD_BASE = f"{XSD_VOLUME_PATH}{XSD_VERSION}/"

# Include paths (same for all IRS forms)
XSD_INCLUDES = ['Common/efileTypes.xsd']

# ============================================================
# FORM PRESETS
# Only form-specific paths - no duplication!
# ============================================================
FORM_PRESETS = {
    '990': {
        'path': 'TEGE/TEGE990/IRS990/IRS990.xsd',
        'root': 'IRS990',
    },
    '990-EZ': {
        'path': 'TEGE/TEGE990EZ/IRS990EZ/IRS990EZ.xsd',
        'root': 'IRS990EZ',
    },
    '990-T': {
        'path': 'TEGE/TEGE990T/IRS990T/IRS990T.xsd',
        'root': 'IRS990T',
    },
    '1040': {
        'path': 'IndividualIncomeTax/Ind1040/IRS1040/IRS1040.xsd',
        'root': 'IRS1040',
    },
    '1120': {
        'path': 'CorporateIncomeTax/Corp1120/IRS1120/IRS1120.xsd',
        'root': 'IRS1120',
    },
}

# ============================================================
# RESOLVE CONFIGURATION
# ============================================================
form_type = dbutils.widgets.get('form_type')

if form_type == 'Custom':
    # Custom form - parse user input
    xsd_override = dbutils.widgets.get('xsd_path_override')
    if not xsd_override:
        raise ValueError("Custom form: provide 'xsd_path_override' as 'path/to/file.xsd|RootElement'")

    parts = xsd_override.split('|')
    CONFIG = {
        'xsd_main_path': parts[0],
        'root_element': parts[1] if len(parts) > 1 else 'Unknown',
        'xsd_include_paths': XSD_INCLUDES,
        'volume_base': XSD_BASE,
    }
else:
    # Use preset
    if form_type not in FORM_PRESETS:
        raise ValueError(f"Unknown form: {form_type}")

    preset = FORM_PRESETS[form_type]
    CONFIG = {
        'xsd_main_path': preset['path'],
        'root_element': preset['root'],
        'xsd_include_paths': XSD_INCLUDES,
        'volume_base': XSD_BASE,
    }

# Add output configuration
CONFIG.update({
    'catalog': dbutils.widgets.get('catalog'),
    'schema': dbutils.widgets.get('schema'),
    'table': dbutils.widgets.get('table'),
    'num_rows': int(dbutils.widgets.get('num_rows')),
    'full_table_name': f"{dbutils.widgets.get('catalog')}.{dbutils.widgets.get('schema')}.{dbutils.widgets.get('table')}"
})

# Display
print("=" * 60)
print(f"Form: {form_type}")
print(f"XSD Version: {XSD_VERSION}")
print(f"XSD Base: {CONFIG['volume_base']}")
print(f"Output: {CONFIG['full_table_name']}")
print(f"Rows: {CONFIG['num_rows']:,}")
print("=" * 60)


## Step 4: Load XSD Schema

In [ ]:
# XSD file paths - from configuration
import os

if 'DATABRICKS_RUNTIME_VERSION' in os.environ:
    # Running in Databricks
    XSD_BASE = CONFIG['volume_base']
    IRS_XSD = f"{XSD_BASE}/{CONFIG['xsd_main_path']}"
    EFILE_TYPES_XSD = [f"{XSD_BASE}/{p}" for p in CONFIG['xsd_include_paths']]
    print(f"XSD: {IRS_XSD}")
else:
    # Running locally - set LOCAL_XSD_PATH environment variable or use ./schemas/
    LOCAL_DIR = os.environ.get('LOCAL_XSD_PATH', './schemas')
    
    # Check if schemas exist
    if not os.path.exists(LOCAL_DIR):
        print(f"WARNING: Schema directory not found: {LOCAL_DIR}")
        print("   Set LOCAL_XSD_PATH environment variable to your IRS schema location")
        print("   Example: export LOCAL_XSD_PATH=/path/to/irs-schemas/2024v5.0")
    
    IRS_XSD = f"{LOCAL_DIR}/{os.path.basename(CONFIG['xsd_main_path'])}"
    EFILE_TYPES_XSD = [f"{LOCAL_DIR}/{os.path.basename(p)}" for p in CONFIG['xsd_include_paths']]
    print(f"Local XSD: {IRS_XSD}")


## Step 5: Generate Synthetic Data

In [ ]:
# Generate synthetic data
df = xsd_to_synthetic(
    spark,
    xsd_path=IRS_XSD,
    root_element=CONFIG['root_element'],
    include_paths=EFILE_TYPES_XSD,
    num_rows=CONFIG['num_rows'],
    flatten_mode='full',  # Always flatten for IRS forms
    null_probability=0.3,
    exclude_root_from_path=True,
    verbose=True
)

# Display sample
print(f"\nGenerated {CONFIG['num_rows']:,} rows with {len(df.columns)} columns")
df.limit(5).display()


## Step 6: Display Results

In [ ]:
# Display schema
df.printSchema()

In [ ]:
# Display sample data
display(df.limit(5))

## Step 7: Write to Unity Catalog

In [ ]:
# Write to Unity Catalog
full_table_name = CONFIG["full_table_name"]
print(f"Writing to: {full_table_name}")

df.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(full_table_name)

print(f"Table created: {full_table_name}")
print(f"  Total rows: {CONFIG['num_rows']:,}")

# Display sample from table
print("\nSample from Unity Catalog table:")
display(spark.table(full_table_name).limit(10))